<a href="https://colab.research.google.com/github/HillaryDrugs/li7/blob/main/LSTM_with_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import re
import random
import os
import time
import math
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ==========================================
# 1. FILE MERGING & PREPROCESSING
# ==========================================
EN_FILE = 'Tatoeba.ar-en.en'
AR_FILE = 'Tatoeba.ar-en.ar'
MERGED_FILE = 'aligned_ara.txt'

# Verify files exist and merge them line-by-line
if os.path.exists(EN_FILE) and os.path.exists(AR_FILE):
    with open(EN_FILE, 'r', encoding='utf-8') as f_en, \
         open(AR_FILE, 'r', encoding='utf-8') as f_ar, \
         open(MERGED_FILE, 'w', encoding='utf-8') as f_out:
        for en_line, ar_line in zip(f_en, f_ar):
            f_out.write(f"{en_line.strip()}\t{ar_line.strip()}\n")
    print(f"Successfully created {MERGED_FILE}")
else:
    raise FileNotFoundError("Please upload Tatoeba.ar-en.en and Tatoeba.ar-en.ar to the sidebar.")

def clean_text(text, is_arabic=False):
    text = str(text).lower().strip()
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    if not is_arabic:
        text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    return text.strip()

class Vocab:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.count = 4
    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.count
                self.idx2word[self.count] = word
                self.count += 1

# Build Vocabularies
df = pd.read_csv(MERGED_FILE, sep='\t', names=['English', 'Arabic'], header=None)
df['English'] = df['English'].apply(lambda x: clean_text(x, False))
df['Arabic'] = df['Arabic'].apply(lambda x: clean_text(x, True))
en_vocab, ar_vocab = Vocab(), Vocab()
for _, row in df.iterrows():
    en_vocab.add_sentence(row['English'])
    ar_vocab.add_sentence(row['Arabic'])

class TranslationDataset(Dataset):
    def __init__(self, dataframe, en_v, ar_v, max_len=25):
        self.df, self.en_v, self.ar_v, self.max_len = dataframe, en_v, ar_v, max_len
    def tokenize(self, text, vocab):
        tokens = [vocab.word2idx.get(w, 3) for w in text.split()][:self.max_len-2]
        return [1] + tokens + [2] + [0] * (self.max_len - len(tokens) - 2)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return torch.tensor(self.tokenize(self.df.iloc[idx]['English'], self.en_v)), \
               torch.tensor(self.tokenize(self.df.iloc[idx]['Arabic'], self.ar_v))

train_df, test_df = train_test_split(df, test_size=0.1)
train_loader = DataLoader(TranslationDataset(train_df, en_vocab, ar_vocab), batch_size=64, shuffle=True)
test_dataset = TranslationDataset(test_df, en_vocab, ar_vocab)

# ==========================================
# 2. MODEL 4 ARCHITECTURE (LSTM + ATTENTION)
# ==========================================
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1)
    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        h = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((h, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    def forward(self, src):
        outputs, (hidden, cell) = self.rnn(self.embedding(src))
        return outputs, hidden, cell

class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.attention = Attention(hid_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(hid_dim + emb_dim, hid_dim, batch_first=True)
        self.out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)
    def forward(self, input, hidden, cell, encoder_outputs):
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)
        rnn_input = torch.cat((self.embedding(input.unsqueeze(1)), weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        prediction = self.out(torch.cat((output, weighted, self.embedding(input.unsqueeze(1))), dim=2).squeeze(1))
        return prediction, hidden, cell

class Seq2SeqAttn(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder, self.decoder, self.device = encoder, decoder, device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        outputs = torch.zeros(batch_size, trg_len, self.decoder.out.out_features).to(self.device)
        enc_out, hidden, cell = self.encoder(src)
        input_step = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_step, hidden, cell, enc_out)
            outputs[:, t] = output
            input_step = trg[:, t] if random.random() < teacher_forcing_ratio else output.argmax(1)
        return outputs

# ==========================================
# 3. TRAINING & BLEU EVALUATION
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Seq2SeqAttn(Encoder(en_vocab.count, 256, 512),
                    AttentionDecoder(ar_vocab.count, 256, 512), DEVICE).to(DEVICE)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)

print(f"Starting Training on {DEVICE}...")
for epoch in range(10): # Increase epochs for better BLEU
    model.train()
    epoch_loss = 0
    for src, trg in train_loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, trg)
        loss = criterion(output[:, 1:].reshape(-1, ar_vocab.count), trg[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

# Final BLEU Calculation
model.eval()
total_bleu = 0
smooth = SmoothingFunction().method1
with torch.no_grad():
    for i in range(200): # Evaluate on 200 samples
        src, trg = test_dataset[i]
        enc_out, hidden, cell = model.encoder(src.unsqueeze(0).to(DEVICE))
        curr_in, res = torch.tensor([1]).to(DEVICE), []
        for _ in range(25):
            out, hidden, cell = model.decoder(curr_in, hidden, cell, enc_out)
            top1 = out.argmax(1)
            if top1.item() == 2: break
            res.append(ar_vocab.idx2word[top1.item()])
            curr_in = top1
        ref = [ar_vocab.idx2word[t.item()] for t in trg if t.item() > 3]
        total_bleu += sentence_bleu([ref], res, smoothing_function=smooth)

print(f"\nFinal Test BLEU Score (Model 4): {total_bleu/200:.4f}")

Successfully created aligned_ara.txt
Starting Training on cuda...
Epoch 1 Loss: 6.0296
Epoch 2 Loss: 4.0943
Epoch 3 Loss: 2.5932
Epoch 4 Loss: 1.6498
Epoch 5 Loss: 1.2704
Epoch 6 Loss: 0.9924
Epoch 7 Loss: 0.7875
Epoch 8 Loss: 0.6429
Epoch 9 Loss: 0.5382
Epoch 10 Loss: 0.4799

Final Test BLEU Score (Model 4): 0.1387
